# Perspective reformulation of a convex cost on a semicontinuous variable
[![perspective_unit_commitment.ipynb](https://img.shields.io/badge/github-%23121011.svg?logo=github)](https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Open In Deepnote](https://deepnote.com/buttons/launch-in-deepnote-small.svg)](https://deepnote.com/launch?url=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Open In Gradient](https://assets.paperspace.io/img/gradient-badge.svg)](https://console.paperspace.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb) [![Powered by AMPL](https://h.ampl.com/https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/modeling-practice/perspective_unit_commitment.ipynb)](https://ampl.com)

Description: Unit commitment with a convex piecewise-linear fuel cost: the perspective reformulation, its linear form for piecewise-linear costs, its form for MP's semicontinuous declarations, and checking its value with the root bound before benchmarking

Tags: modeling-tips, mp, perspective, semi-continuous, unit-commitment, piecewise-linear, reinforced-formulations, formulation-comparison, milp, amplpy

<!--
Domain: Energy
Subdomain: Power Generation
Model type: MILP
Level: Lecture
Algorithm: Perspective Reformulation
Subject: Unit Commitment
-->

Notebook author: Gleb Belov <<gleb@ampl.com>>

References:
1. Frangioni, A., Gentile, C. (2006). Perspective cuts for a class of convex 0-1 mixed integer programs. *Mathematical Programming* 106(2), 225-236.
2. Günlük, O., Linderoth, J. (2010). Perspective reformulations of mixed integer nonlinear programs with indicator variables. *Mathematical Programming* 124, 183-205.
3. MP modeling guide, *Choosing between formulations*: https://mp.ampl.com/modeling-formulations.html

In [1]:
# Install dependencies
%pip install -q amplpy numpy pandas

In [2]:
# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=["highs"],  # modules to install
    license_uuid="default",  # license to use
)  # instantiate AMPL object and register magics

In [3]:
SOLVER_MILP = (
    "highs"  # any MP-based MILP solver: highs, gurobi, xpress, cplex, copt, scip, ...
)

## The structure

A generator is either off ($P = 0$) or on with $p^{\min} \le P \le p^{\max}$:
a **semicontinuous** variable. Its fuel cost $f(P)$ is convex, with $f(0) = 0$,
and paid only while it is on. The textbook formulation uses an on/off binary $Z$:

$$p^{\min} Z \le P \le p^{\max} Z, \qquad \text{Cost} \ge f(P).$$

It is exact, but its LP relaxation is weak: at a fractional $Z$ the cost row
says nothing about $Z$. The **perspective** of $f$,

$$\text{Cost} \ge Z \, f(P / Z),$$

is exact at $Z \in \{0, 1\}$ and, together with the on/off bounds, describes the
convex hull of "off, or on and paying $f$" (Frangioni & Gentile 2006).

You almost never write $Z f(P/Z)$ literally:

* **Piecewise-linear convex $f$** $= \max_k (s_k P + c_k)$: the perspective is the
  same rows with every **intercept multiplied by $Z$**,
  $\text{Cost} \ge s_k P + c_k Z$. Linear, no new variables.
* **Quadratic $f = q P^2$**: the perspective is a rotated second-order cone,
  $q P^2 \le \text{Cost} \cdot Z$.

On a convex ladder through the origin, every $c_k$ with $k \ge 2$ is negative, so
a fractional $Z$ raises the right-hand side. That is the whole tightening, and
it vanishes at integrality.

## Data

In [4]:
import time
import numpy as np
import pandas as pd


def make_data(n_units, n_periods, n_seg=3, seed=7):
    rng = np.random.default_rng(seed)
    pmin = rng.integers(30, 80, n_units)
    pmax = pmin + rng.integers(80, 200, n_units)
    base = rng.uniform(8, 20, n_units)
    slope = {
        (u, k): float(round(base[u] + (k - 1) * (4 + 4 * rng.random()), 2))
        for u in range(n_units)
        for k in range(1, n_seg + 1)
    }  # increasing: convex
    brk = {
        (u, k): float(round(pmin[u] + (pmax[u] - pmin[u]) * k / n_seg, 1))
        for u in range(n_units)
        for k in range(1, n_seg)
    }
    demand = (
        0.6
        * pmax.sum()
        * (0.55 + 0.45 * np.sin(np.arange(n_periods) / n_periods * 2 * np.pi) ** 2)
    )
    return dict(
        n_seg=n_seg,
        nT=n_periods,
        pmin=pmin.tolist(),
        pmax=pmax.tolist(),
        fix=rng.integers(100, 600, n_units).tolist(),
        su=rng.integers(300, 1500, n_units).tolist(),
        slope=slope,
        brk=brk,
        dem={t + 1: float(round(d)) for t, d in enumerate(demand)},
    )

## Four exact formulations

* `binary_pl`: on/off binary, cost as AMPL's piecewise-linear term `<<breakpoints; slopes>>`.
* `binary_epigraph`: on/off binary, cost as max-of-affine rows (no perspective).
* `binary_perspective`: on/off binary, intercepts multiplied by `On`.
* `semicontinuous_perspective`: no binary in the model. `P` is declared
  `in {0} union interval[pmin, pmax]`, and the intercepts are gated by an `if`
  on the threshold `pmin/2`, which lies inside the gap $(0, p^{\min})$, so
  `P >= pmin/2` is exactly "on".

In [5]:
COMMON = r"""
set U; param nT; set T := 1..nT; param n_seg;
param pmin {U}; param pmax {U}; param fix {U}; param su {U};
param slope {U, 1..n_seg}; param brk {U, 1..n_seg-1}; param dem {T};
# intercepts of the max-of-affine form of the convex ladder (c[u,1] = 0)
param c {u in U, k in 1..n_seg} := sum {m in 1..k-1} (slope[u,m] - slope[u,m+1]) * brk[u,m];
var Start {U, T} binary;
var Cost {U, T} >= 0;
"""
BINARY = r"""
var On {U, T} binary;
var P {u in U, T} >= 0, <= pmax[u];
subject to Lo {u in U, t in T}: P[u,t] >= pmin[u] * On[u,t];
subject to Hi {u in U, t in T}: P[u,t] <= pmax[u] * On[u,t];
"""
SEMICONT = r"""
var P {u in U, T} in {0} union interval[pmin[u], pmax[u]];
var On {u in U, t in T} = if P[u,t] >= pmin[u]/2 then 1;
"""
REST = r"""
minimize Total: sum {u in U, t in T} (Cost[u,t] + fix[u] * On[u,t] + su[u] * Start[u,t]);
subject to Demand {t in T}: sum {u in U} P[u,t] >= dem[t];
subject to Start_Def {u in U, t in T: t > 1}: Start[u,t] >= On[u,t] - On[u,t-1];
"""
MODELS = {
    "binary_pl": COMMON
    + BINARY
    + REST
    + r"""
subject to Cost_PL {u in U, t in T}:
   Cost[u,t] >= <<{k in 1..n_seg-1} brk[u,k]; {k in 1..n_seg} slope[u,k]>> P[u,t];
""",
    "binary_epigraph": COMMON
    + BINARY
    + REST
    + r"""
subject to Cost_Seg {u in U, t in T, k in 1..n_seg}: Cost[u,t] >= slope[u,k] * P[u,t] + c[u,k];
""",
    "binary_perspective": COMMON
    + BINARY
    + REST
    + r"""
subject to Cost_Persp {u in U, t in T, k in 1..n_seg}:
   Cost[u,t] >= slope[u,k] * P[u,t] + c[u,k] * On[u,t];
""",
    "semicontinuous_perspective": COMMON
    + SEMICONT
    + REST
    + r"""
subject to Cost_Seg1 {u in U, t in T}: Cost[u,t] >= slope[u,1] * P[u,t];
subject to Cost_Persp {u in U, t in T, k in 2..n_seg}:
   Cost[u,t] >= slope[u,k] * P[u,t] + (if P[u,t] >= pmin[u]/2 then c[u,k]);
""",
}


def load(model, data):
    ampl = AMPL()
    ampl.eval(model)
    ampl.set["U"] = range(len(data["pmin"]))
    ampl.param["nT"] = data["nT"]
    ampl.param["n_seg"] = data["n_seg"]
    for name in ("pmin", "pmax", "fix", "su"):
        ampl.param[name] = dict(enumerate(data[name]))
    for name in ("slope", "brk", "dem"):
        ampl.param[name] = data[name]
    return ampl


def solver_objective(ampl):
    """The objective value the solver reports (last 'objective <value>' in solve_message).

    AMPL's own value, ampl.obj["Total"].value(), is recomputed from the returned
    variables. For a relaxation of a model with logical expressions (if-then, ==>)
    the two differ: AMPL evaluates the logic exactly at a fractional point."""
    import re

    found = re.findall(r"objective ([-+0-9.eE]+)", ampl.get_value("solve_message"))
    return float(found[-1]) if found else ampl.obj["Total"].value()


def solve(model, data, relax=False, solver_options=""):
    ampl = load(model, data)
    opts = "outlev=0 " + solver_options + (" relax=1 chk:mode=0" if relax else "")
    start = time.time()
    ampl.solve(solver=SOLVER_MILP, verbose=False, **{SOLVER_MILP + "_options": opts})
    return dict(
        status=ampl.solve_result,
        objective=solver_objective(ampl),
        seconds=round(time.time() - start, 2),
    )

## Check the tightening is not empty: compare root bounds

This takes seconds and should come before any benchmark. Two details matter
when the model contains logic that MP converts:

* Use the solver option `relax=1`, which relaxes integrality of **all** variables
  the solver receives, including the binaries MP creates for the semicontinuous
  declaration and the `if`. AMPL's `option relax_integrality 1` relaxes only the
  model's own integer variables, so with MP-converted logic it does not give the
  root bound.
* Read the bound the **solver** reports. After a relaxed solve, AMPL recomputes
  `Total` from the returned variables and evaluates `if P >= pmin/2 then ...`
  exactly at the fractional point, which can give a value above the true
  optimum. MP's solution check warns about this (`expr '_ifthen'`). The helper
  `solver_objective` above takes the value from the solve message.

In [6]:
def compare(data):
    rows = []
    for name, model in MODELS.items():
        root = solve(model, data, relax=True)
        mip = solve(model, data, solver_options="timelim=300")
        rows.append(
            dict(
                formulation=name,
                root_bound=round(root["objective"], 1),
                optimum=round(mip["objective"], 1),
                status=mip["status"],
                seconds=mip["seconds"],
            )
        )
    table = pd.DataFrame(rows).set_index("formulation")
    table["root_gap_%"] = (
        100 * (table.optimum - table.root_bound) / table.optimum
    ).round(3)
    return table


for n_units, n_periods in [(10, 48), (20, 96)]:
    print(f"\n{n_units} units x {n_periods} periods")
    display(compare(make_data(n_units, n_periods)))


10 units x 48 periods


,root_bound,optimum,status,seconds,root_gap_%
formulation,,,,,
binary_pl,676889.1,711374.8,solved,0.34,4.848
binary_epigraph,676889.1,711374.8,solved,1.01,4.848
binary_perspective,711143.2,711374.8,solved,0.51,0.033
semicontinuous_perspective,711143.2,711374.8,solved,0.54,0.033



20 units x 96 periods


,root_bound,optimum,status,seconds,root_gap_%
formulation,,,,,
binary_pl,2427424.7,2592323.0,solved,1.89,6.361
binary_epigraph,2427424.7,2592323.0,solved,6.57,6.361
binary_perspective,2591724.1,2592323.0,solved,5.06,0.023
semicontinuous_perspective,2591724.1,2592323.0,solved,5.43,0.023


Read the table in two parts.

**Root bound.** The perspective closes nearly all of the root gap that the
textbook formulations leave, and the semicontinuous form with gated intercepts
has exactly the same root bound: MP maps the threshold test onto the binary it
creates for the declaration, so it is the same perspective. The piecewise-linear and epigraph forms have identical root
bounds: they describe the same function, just written differently.

**Time.** A much tighter relaxation does not have to be faster. The solver's
own presolve, cuts and heuristics can work better on one form than another, and
the ranking can change with the solver and the instance size. Set
`SOLVER_MILP` to another solver and rerun to see how your solver behaves. The
root gap is the solver-independent part; the time is what you measure.

All four reach the same optimum, as exact reformulations must, up to the
solver's relative MIP gap tolerance. Checking this on every instance you
benchmark is the cheapest validation of a reformulation there is.

## Where the perspective does not apply

* A **fixed cost** paid while on, $\text{fix} \cdot Z$, already is a perspective:
  do not add it twice.
* A **linear** cost has no intercepts to multiply; there is nothing to gain.
* A **concave** cost has no useful perspective. Within a disjunction, use a
  secant over the on-range instead.
* Do not convert perspective rows to indicators
  (`On = 1 ==> Cost >= slope*P + c`): the indicator holds only at integrality and
  discards exactly the strengthening the rows provide at fractional `On`.

## Takeaways

* For a convex cost on a semicontinuous variable, write the perspective. For
  piecewise-linear costs it is linear: multiply each intercept by the on/off
  binary, or gate it with `if` on a threshold inside the gap when the variable is
  declared semicontinuous.
* Compare root bounds before benchmarking (`relax=1`, or `option
  relax_integrality 1` for the model's own integer variables). A tightening
  that does not move the root bound is not worth a benchmark.
* Measure time on the solvers you will use; a tighter relaxation can be slower.

Related notebooks: https://colab.ampl.com/tags/reinforced-formulations.html and
https://colab.ampl.com/tags/formulation-comparison.html.